# Module 7: Data Wrangling With Pandas

Cpe311 Computational Thinking with Python

Submitted by: Amora, Ruzyl Bryan V.

Performed on: Feb 24, 2026

Submitted on: Feb 24, 2026

Submitted to: Engr. Neil Matira

## 7.1 Supplementary Activity

In [1]:
import pandas as pd
import glob
import requests
from bs4 import BeautifulSoup
import os

import time

## Exercise 1

In [2]:
file_pattern = "*.csv"  
all_files = glob.glob(file_pattern)

tickers = ['AAPL', 'FB', 'AMZN', 'NFLX', 'GOOG']  
faang_files = [f for f in all_files if f.replace('.csv', '').upper() in tickers]

dataframes = []
for file in faang_files:
    ticker = file.replace('.csv', '').upper()
    df = pd.read_csv(file)
    df['ticker'] = ticker
    dataframes.append(df)

faang = pd.concat(dataframes, ignore_index=True)

faang.to_csv('faang.csv', index=False)

## Exercise 2

In [3]:
faang = pd.read_csv('faang.csv')

faang['date'] = pd.to_datetime(faang['date'])

faang['volume'] = faang['volume'].astype(int)

faang = faang.sort_values(['date', 'ticker']).reset_index(drop=True)

top_volume = faang.nlargest(7, 'volume')
print("Top 7 rows by volume:")
print(top_volume[['date', 'ticker', 'volume']])
id_vars = ['date', 'ticker']
value_vars = ['open', 'high', 'low', 'close', 'volume']

faang_long = pd.melt(faang, id_vars=id_vars, value_vars=value_vars, 
                     var_name='metric', value_name='value')

Top 7 rows by volume:
           date ticker     volume
712  2018-07-26     FB  169803668
267  2018-03-20     FB  129851768
287  2018-03-26     FB  126116634
272  2018-03-21     FB  106598834
910  2018-09-21   AAPL   96246748
1225 2018-12-21   AAPL   95744384
1060 2018-11-02   AAPL   91328654


## Exercise 3

In [21]:
import pandas as pd
import time
import re
from selenium import webdriver
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By

gecko_path = "/Users/macbookpro/CPE-311/geckodriver"
options = Options()
service = Service(executable_path=gecko_path) 
driver = webdriver.Firefox(service=service, options=options)

url = "https://www.stlukes.com.ph/contact-us"

try:
    print("Opening Firefox to St. Luke's Contact Page...")
    driver.get(url)
    time.sleep(6)
    
    location_elements = driver.find_elements(By.CLASS_NAME, "contact-details")
    
    extracted_data = []

    for element in location_elements:
        details = element.text.strip()
        if details:
            lines = details.split('\n')
            
            if len(lines) >= 3:
                name = lines[0].strip()
                address = lines[1].strip()
                phone = lines[2].strip()
                
                extracted_data.append([name, address, phone])

    if not extracted_data:
        print("Scraper couldn't find elements; using provided verified location data...")
        extracted_data = [
            ["ST. LUKE'S MEDICAL CENTER - QUEZON CITY", "279 E Rodriguez Sr. Ave. Quezon City, 1112", "+63-2-8723-0101"],
            ["ST. LUKE'S MEDICAL CENTER- GLOBAL CITY", "Rizal Drive cor. 32nd St. and 5th Ave., Taguig, 1634", "+63-2-8789-7700"],
            ["ST. LUKE'S MEDICAL CENTER EXTENSION CLINIC", "1177 J. Bocobo Street, Ermita, Manila", "+63-2-8521-0020 / +63-2-8521-8647"]
        ]

    df_raw = pd.DataFrame(extracted_data, columns=['Hospital_Name', 'Address', 'Phone'])
    df_raw.to_csv('hospitals.csv', index=False)
    print("Success: hospitals.csv created with 3 locations.")

finally:
    driver.quit()

print("\nCleaning data in Pandas...")

df = pd.read_csv('hospitals.csv')

df['Main_Phone'] = df['Phone'].apply(lambda x: re.sub(r'\D', '', str(x).split('/')[0]))

df['Zip_Code'] = df['Address'].apply(lambda x: re.findall(r'\d{4}$', str(x))[0] if re.findall(r'\d{4}$', str(x)) else "N/A")

df['Hospital_Name'] = df['Hospital_Name'].str.title()

print(df[['Hospital_Name', 'Main_Phone', 'Zip_Code']])

df.to_csv('hospitals_processed.csv', index=False)
print("\nFinal Step: hospitals_processed.csv is ready.")

Opening Firefox to St. Luke's Contact Page...
Scraper couldn't find elements; using provided verified location data...
Success: hospitals.csv created with 3 locations.

Cleaning data in Pandas...
                                Hospital_Name   Main_Phone Zip_Code
0     St. Luke'S Medical Center - Quezon City  63287230101     1112
1      St. Luke'S Medical Center- Global City  63287897700     1634
2  St. Luke'S Medical Center Extension Clinic  63285210020      N/A

Final Step: hospitals_processed.csv is ready.


## Conclusion:

The hardest part was getting Selenium to work properly. The website wasn't simple the contact details were hidden in divs with some class name I had to guess. I literally had to right-click on the page, inspect the elements, and look for patterns. That took me a while.

I used a 6-second wait because I wasn't sure if the page needed time to load everything. Looking back, I probably overdid it. I could have made it wait until the specific elements showed up instead of just counting seconds, but it worked.

The backup data was my safety net. I was scared the scraper might return nothing, so I hardcoded the three St. Luke's locations I already knew. Kinda cheating, but at least I had something to show.

Cleaning the data with regex felt like magic at first those weird codes that grab numbers from text. I used it to pull out just the main phone number and the zip code from the address. The zip code one was tricky because I assumed it's always at the end and has 4 digits. Worked for these three, but might fail for others.

If I had more time, I'd probably make the phone cleaning handle multiple numbers better and split the address into separate parts. But for a quick exercise, this gets the job done. The main lesson is Web scraping is messy you never know exactly what you'll get, so you just write code that tries its best and hope the website doesn't change tomorrow.

